# Process plate counts to get ratios of variants in a repooled library

Based on the transcriptional titers of each strain in the initial, equal volume pool, I repooled all library strains (dropping those with lowest titer) and then infected this pool on MDCK-SIAT1 cells. These infections were done by serially 1:2 diluting a starting 1:4 dilution  of pool. Barcodes were then isolated and sequenced so that we can determine representation of each variant in the pool, as well as the appropriate library pool dilution (i.e., MOI) to use in neutralization assays. 

The plots generated by this notebook are interactive, so you can mouseover points for details, use the mouse-scroll to zoom and pan, and use interactive dropdowns at the bottom of the plots.

## Setup
Import Python modules:

In [1]:
import pickle
import sys

import altair as alt

import numpy
import string
import pandas as pd
from os.path import join
import os
import ruamel.yaml as yaml

_ = alt.data_transformers.disable_max_rows()

# Basic color palette
color_palette = [
    '#345995', #blue
    '#03cea4', #teal
    '#ca1551', #red
    '#eac435', #yellow
               ]

In [2]:
resultsdir = '../results'
os.makedirs(resultsdir, exist_ok=True)

## Add input data locations
Some of these files are defined as data, and some of these files are generated by running the specified library pooling data as `miscellaneous_plates` through the `seqneut-pipeline`. For details on how these files are generated, see the `README.md' in [https://github.com/jbloomlab/seqneut-pipeline](https://github.com/jbloomlab/seqneut-pipeline)

In [3]:
# Viral library contents and barcode IDs
viral_library_csv =  '../../../data/viral_libraries/flu-seqneut-2025to2026-barcode-to-strain-actual.csv'
# Neutralization standard set of barcode IDs
neut_standard_set_csv =  '../../../data/neut_standard_sets/loes2023_neut_standards.csv'
# All samples included in library poolign sequencing run
# Contains information on library, dilution factor, R1 location
samplesfile = '../../../data/miscellaneous_plates/2026-01-22_repool.csv'

## CHANGE THIS EACH TIME YOU RUN NB ##
# Counts and fates files output by running library pooling samples as miscellaneous plates
platedir = '../../../results/miscellaneous_plates/20260122_repool/'

# Identify all counts and fates CSVs
count_csvs = []
fate_csvs = []
file_list = os.listdir(platedir)
for f in file_list:
    location = platedir + f
    if "_counts" in f:
        count_csvs.append(location)
    elif "_fates" in f:
        fate_csvs.append(location)

In [4]:
# Define a samples dataframe using the samples file
samples_df = pd.read_csv(samplesfile)
samples_df.drop(columns=['fastq'], inplace=True)
samples_df['sample'] = samples_df.apply(
    lambda x: '-'.join(x.astype(str)), axis=1
)

samples = samples_df["sample"].unique().tolist()
print(f"There are {len(samples)} barcode runs.")

samples_df

There are 16 barcode runs.


,well,serum,dilution_factor,replicate,sample
0,A1,none,4,1,A1-none-4-1
1,B1,none,8,1,B1-none-8-1
2,C1,none,16,1,C1-none-16-1
3,D1,none,32,1,D1-none-32-1
4,E1,none,64,1,E1-none-64-1
5,F1,none,128,1,F1-none-128-1
6,G1,none,256,1,G1-none-256-1
7,H1,none,512,1,H1-none-512-1
8,A2,none,4,2,A2-none-4-2
9,B2,none,8,2,B2-none-8-2


## Statistics on barcode-parsing for each sample
Make interactive chart of the "fates" of the sequencing reads parsed for each sample on the plate.

If most sequencing reads are not "valid barcodes", this could potentially indicate some problem in the sequencing or barcode set you are parsing.

Potential fates are:
 - *valid barcode*: barcode that matches a known virus or neutralization standard, we hope most reads are this.
 - *invalid barcode*: a barcode with proper flanking sequences, but does not match a known virus or neutralization standard. If you  have a lot of reads of this type, it is probably a good idea to look at the invalid barcode CSVs (in the `./results/barcode_invalid/` subdirectory created by the pipeline) to see what these invalid barcodes are.
 - *unparseable barcode*: could not parse a barcode from this read as there was not a sequence of the correct length with the appropriate flanking sequence.
 - *low quality barcode*: low-quality or `N` nucleotides in barcode, could indicate problem with sequencing.
 - *failed chastity filter*: reads that failed the Illumina chastity filter, if these are reported in the FASTQ (they may not be).

Also, if the number of reads per sample is very uneven, that could indicate that you did not do a good job of balancing the different samples in the Illumina sequencing.

In [5]:
fates = (
    pd.concat([pd.read_csv(f).assign(well=f.strip(platedir).strip('_fates.csv')) for f, s in zip(fate_csvs, samples)])
    .merge(samples_df, validate="many_to_one", on="well")
    .assign(
        fate_counts=lambda x: x.groupby("fate")["count"].transform("sum"),
        sample_well=lambda x: x["sample"] + " (" + x["well"] + ")",
    )
    .query("fate_counts > 0")[  # only keep fates with at least one count
        ["fate", "count", "well", "sample_well", "dilution_factor"]
    ]
)

assert len(fates) == len(fates.drop_duplicates())


sample_wells = list(
    fates.sort_values(["dilution_factor"])["sample_well"]
)



fates_chart = (
    alt.Chart(fates)
    .encode(
        alt.X("count", scale=alt.Scale(nice=False, padding=3)),
        alt.Y(
            "sample_well",
            title=None,
            sort=sample_wells,
        ),
        alt.Color("fate", sort=sorted(fates["fate"].unique(), reverse=True)),
        alt.Order("fate", sort="descending"),
        tooltip=fates.columns.tolist(),
    )
    .mark_bar(height={"band": 0.85})
    .properties(
        height=alt.Step(10),
        width=200,
        title=f"Barcode parsing for initial titering plate",
    )
    .configure_axis(grid=False)
)

fates_chart

alt.Chart(...)

## Read barcode counts
Read the counts per barcode:

In [6]:
# get barcode counts
counts = (
    pd.concat([pd.read_csv(c).assign(well=c.replace(platedir,'').strip('_counts.csv')) for c, s in zip(count_csvs, samples)])
    .merge(samples_df, validate="many_to_one", on="well")
    .drop(columns=["replicate"])
    .assign(sample_well=lambda x: x["sample"] + " (" + x["well"] + ")")
)

# classify barcodes as viral or neut standard
barcode_class = pd.concat(
    [
        pd.read_csv(viral_library_csv)[["barcode", "strain"]].assign(
            neut_standard=False,
        ),
        pd.read_csv(neut_standard_set_csv)[["barcode"]].assign(
            neut_standard=True,
            strain=pd.NA,
        ),
    ],
    ignore_index=True,
)

counts

# merge counts and classification of barcodes
assert set(counts["barcode"]) == set(barcode_class["barcode"])
counts = counts.merge(barcode_class, on="barcode", validate="many_to_one")
assert set(sample_wells) == set(counts["sample_well"])

In [7]:
counts

,barcode,count,fraction_all_valid_and_invalid_counts,well,serum,dilution_factor,sample,sample_well,strain,neut_standard
0,CAATTAGAAATACATA,1946210,0.148,H2,none,512,H2-none-512-2,H2-none-512-2 (H2),NaN,True
1,GTACAAACCTGCAAAT,1880441,0.143,H2,none,512,H2-none-512-2,H2-none-512-2 (H2),NaN,True
2,AAAAAATTTATGACAA,1713864,0.130,H2,none,512,H2-none-512-2,H2-none-512-2 (H2),NaN,True
3,TACCCTGCAAGCCACT,1586904,0.120,H2,none,512,H2-none-512-2,H2-none-512-2 (H2),NaN,True
4,AACCACCGAGTGACCG,1330720,0.101,H2,none,512,H2-none-512-2,H2-none-512-2 (H2),NaN,True
...,...,...,...,...,...,...,...,...,...,...
3483,TAGACTGGATACGTGA,0,0.000,A2,none,4,A2-none-4-2,A2-none-4-2 (A2),A/Catalonia/NSAV198331894/2025_H1N1,False
3484,TCCGCACTGCGATCAC,0,0.000,A2,none,4,A2-none-4-2,A2-none-4-2 (A2),A/DistrictOfColumbia/27/2023_H3N2,False
3485,TCTTCAAGTCGTGTTA,0,0.000,A2,none,4,A2-none-4-2,A2-none-4-2 (A2),A/England/1845724/2025_H3N2,False
3486,TGGACTGGCCATCCTA,0,0.000,A2,none,4,A2-none-4-2,A2-none-4-2 (A2),A/SuratThani/P3371/2025_H3N2,False


## Average counts per barcode in each well

Plot average counts per barcode.
If a sample has inadequate barcode counts, it may not have good enough statistics for accurate analysis, and a QC-threshold is applied:

In [8]:
avg_barcode_counts = (
    counts.groupby(
        ["well", "sample_well"],
        dropna=False,
        as_index=False,
    )
    .aggregate(avg_count=pd.NamedAgg("count", "mean"))
    .assign(
        fails_qc=lambda x: (
            x["avg_count"] < 500
        ),
    )
)

avg_barcode_counts_chart = (
    alt.Chart(avg_barcode_counts)
    .encode(
        alt.X(
            "avg_count",
            title="average barcode counts per well",
            scale=alt.Scale(nice=False, padding=3),
        ),
        alt.Y("sample_well", sort=sample_wells),
        alt.Color(
            "fails_qc",
            title=f"fails {'min barcode count threshold'=}",
            legend=alt.Legend(titleLimit=500),
        ),
        tooltip=[
            alt.Tooltip(c, format=".3g") if avg_barcode_counts[c].dtype == float else c
            for c in avg_barcode_counts.columns
        ],
    )
    .mark_bar(height={"band": 0.85})
    .properties(
        height=alt.Step(10),
        width=250,
        title=f"Average barcode counts per well for titering plate",
    )
    .configure_axis(grid=False)
)

display(avg_barcode_counts_chart)

# drop wells failing QC
avg_barcode_counts_per_well_drops = list(avg_barcode_counts.query("fails_qc")["well"])

alt.Chart(...)

## Fraction of counts from neutralization standard
Determine the fraction of counts from the neutralization standard in each sample, and make sure this fraction passess the QC threshold.

In [9]:
neut_standard_fracs = (
    counts.assign(
        neut_standard_count=lambda x: x["count"] * x["neut_standard"].astype(int)
    )
    .groupby(
        ["well", "sample_well", 'dilution_factor'],
        dropna=False,
        as_index=False,
    )
    .aggregate(
        total_count=pd.NamedAgg("count", "sum"),
        neut_standard_count=pd.NamedAgg("neut_standard_count", "sum"),
    )
    .assign(
        neut_standard_frac=lambda x: x["neut_standard_count"] / x["total_count"],
        fails_qc=lambda x: (
            x["neut_standard_frac"] < 0.001
        ),
    )
)

neut_standard_fracs_chart = (
    alt.Chart(neut_standard_fracs)
    .encode(
        alt.X(
            "neut_standard_frac",
            title="frac counts from neutralization standard per well",
            scale=alt.Scale(nice=False, padding=3),
        ),
        alt.Y("sample_well", sort=sample_wells),
        alt.Color(
            "fails_qc",
            title=f"fails {'min_neut_standard_frac_per_well'=}",
            legend=alt.Legend(titleLimit=500),
        ),
        tooltip=[
            alt.Tooltip(c, format=".3g") if neut_standard_fracs[c].dtype == float else c
            for c in neut_standard_fracs.columns
        ],
    )
    .mark_bar(height={"band": 0.85})
    .properties(
        height=alt.Step(10),
        width=250,
        title=f"Neutralization-standard fracs per well for titering plate, initial pool",
    )
    .configure_axis(grid=False)
    .configure_legend(titleLimit=1000)
)

display(neut_standard_fracs_chart)

# drop wells failing QC
min_neut_standard_frac_per_well_drops = list(
    neut_standard_fracs.query("fails_qc")["well"]
)

alt.Chart(...)

In [10]:
# Scatterplot of the same data as above, plotted by dilution factor
alt.Chart(neut_standard_fracs).mark_circle(size=60).encode(
    alt.X('dilution_factor:Q', 
          scale=alt.Scale(type='log'),
          title='library pool reciprocal dilution factor'),
    alt.Y('neut_standard_frac:Q', 
          scale=alt.Scale(type='log'),
          title='fraction of reads = neutralization standard'),
    color='fails_qc',
    tooltip=['well', 'dilution_factor', 'neut_standard_frac', 'total_count']
).interactive()

alt.Chart(...)

## Assess balancing of strains contained in the library

In [11]:
# Get summed barcode counts for all strains across all wells
straincounts_allbarcodes = (counts.groupby(['sample','sample_well','strain','dilution_factor','serum','well'])
                          .sum()
                          .reset_index()
                          .drop(columns = ['sample_well', 'neut_standard', 'barcode'])
                         )

# Get sum of all virus/barcode counts per well
sumperwell = (straincounts_allbarcodes.groupby(['sample','dilution_factor','serum','well'])
              .sum()
              .drop(columns=['strain'])
              .reset_index()
              .rename(columns={'count':'counts_perwell'})
             )

# Merge dataframes and calculate fraction of each well devoted to each strain
merged_df = straincounts_allbarcodes.merge(sumperwell, on=['sample','dilution_factor','serum','well'])
merged_df['fraction_strain'] = merged_df['count'] /merged_df['counts_perwell'] / 2
merged_df

,sample,strain,dilution_factor,serum,well,count,fraction_all_valid_and_invalid_counts_x,counts_perwell,fraction_all_valid_and_invalid_counts_y,fraction_strain
0,A1-none-4-1,A/Aberystwyth/5654/2025_H3N2,4,none,A1,0,0.000000,17805445,0.740856,0.000000
1,A1-none-4-1,A/Alagoas/4130/2025_H3N2,4,none,A1,113346,0.004720,17805445,0.740856,0.003183
2,A1-none-4-1,A/Anapolis/LEIAL6543/2025_H3N2,4,none,A1,81827,0.003400,17805445,0.740856,0.002298
3,A1-none-4-1,A/Andalucia/PMC-00977/2025_H1N1,4,none,A1,284984,0.011860,17805445,0.740856,0.008003
4,A1-none-4-1,A/Aragon/2046/2025_H1N1,4,none,A1,213753,0.008890,17805445,0.740856,0.006002
...,...,...,...,...,...,...,...,...,...,...
1611,H2-none-512-2,A/Vermont/73/2025_H3N2,512,none,H2,0,0.000000,125692,0.009523,0.000000
1612,H2-none-512-2,A/Victoria/4897/2022_IVR-238_H1N1,512,none,H2,0,0.000000,125692,0.009523,0.000000
1613,H2-none-512-2,A/Wisconsin/588/2019_H1N1,512,none,H2,862,0.000065,125692,0.009523,0.003429
1614,H2-none-512-2,A/Wisconsin/67/2022_H1N1,512,none,H2,2882,0.000218,125692,0.009523,0.011465


We now have this fraction of reads devoted to all strains calculated for all wells. However, ideally we should just focus on those wells containing dilutions that we would use for actual neutralization assays. We should choose a set of replicate wells where the fraction of neutralization standard reads begins to increase linearly with the increasing reciprocal dilution factor. See plots above for choosing these wells. 

In [12]:
# Choose wells
well1 = 'C1'
well2 = 'C2'

In [13]:
# Choose a pair of replicate wells near the beginning of the linear range
single_well = merged_df.loc[merged_df['sample'].str.contains(f'{well1}-|{well2}-')]

In [14]:
# Calculate mean fraction strain across both wells
mean_df = single_well.groupby(['strain'])['fraction_strain'].mean().to_frame().rename(columns = {'fraction_strain': 'mean_fraction_strains'}).reset_index()
mean_single_well = single_well.merge(mean_df, on = 'strain', how = 'left')

# calcualte ratios to add for equal pool
num_strains = len(mean_single_well.strain.unique())
mean_single_well['ratio_to_add'] = (1/num_strains)/mean_single_well['fraction_strain']
mean_single_well['mean_ratio_to_add'] = (1/num_strains)/mean_single_well['mean_fraction_strains']

mean_single_well['est_tcid50'] = (mean_single_well['mean_fraction_strains']*25000)*76

print(f'this library has {num_strains} total strains')
print('stats where there isnt 0 of a virus...')
print(mean_single_well.query('mean_ratio_to_add != inf')[['mean_ratio_to_add']].describe())

print('\nviruses with 0 titer...')
print(mean_single_well.query('mean_ratio_to_add == inf').strain.unique())

ratio_cutoff = 250
print(f'\nviruses with >0 titer but ratio >={ratio_cutoff} to increase...')
print(mean_single_well.query('mean_ratio_to_add != inf').query(f'mean_ratio_to_add >= {ratio_cutoff}').strain.unique())

this library has 101 total strains
stats where there isnt 0 of a virus...
       mean_ratio_to_add
count         182.000000
mean            3.033394
std             3.596762
min             0.487969
25%             1.498086
50%             2.142263
75%             2.799275
max            27.942964

viruses with 0 titer...
['A/Aberystwyth/5654/2025_H3N2' 'A/Bahrain/4684/2025_H3N2'
 'A/BritishColumbia/PHL-2467/2025_H1N1'
 'A/Catalonia/NSAV198331894/2025_H1N1' 'A/Delaware/81/2025_H3N2'
 'A/England/1845724/2025_H3N2' 'A/Galicia/GA-CHUAC-449/2025_H1N1'
 'A/Malaysia/IMR-SARI1798/2025_H1N1'
 'A/SouthAustralia/2523011822/2025_H1N1' 'A/SuratThani/P3371/2025_H3N2']

viruses with >0 titer but ratio >=250 to increase...
[]


## Re-pooling calculations

In [15]:
# Factor to multiply ratios by
repool_factor = 25

In [16]:
# Get library IDs
lib_id_df=pd.read_csv('../../library_design/construct_order/flu-seqneut-2025to2026-barcode-to-strain-designed.csv')

# Get old strain metadata
h3_metadata_df=pd.read_csv('../../library_design/initial_design/data/2025-NH-VCM-neutralization-library-strain-selection-H3N2.tsv', sep='\t')
h1_metadata_df=pd.read_csv('../../library_design/initial_design/data/2025-NH-VCM-neutralization-library-strain-selection-H1N1.tsv', sep='\t')

h3_metadata_df['subtype'] = 'H3N2'
h1_metadata_df['subtype'] = 'H1N1'
h3_metadata_df['strain'] = h3_metadata_df['representative_strain'] + '_' + h3_metadata_df['subtype']
h1_metadata_df['strain'] = h1_metadata_df['representative_strain'] + '_' + h1_metadata_df['subtype']

metadata_df = pd.concat([h3_metadata_df, h1_metadata_df])
metadata_df = metadata_df.rename(columns={
    'derived_haplotype': 'old_derived_haplotype',
    'count': 'old_count',
    'latest_sequence': 'old_latest_sequence',
})

# Get new strain metadata
new_h3_metadata_df=pd.read_csv('../data/h3n2_haplotypes_updated_from_John_2026-01-09.tsv', sep='\t')
new_h1_metadata_df=pd.read_csv('../data/h1n1pdm_haplotypes_updated_from_John_2026-01-09.tsv', sep='\t')
new_metadata_df = pd.concat([new_h3_metadata_df, new_h1_metadata_df])

# # Merge metadata on HA sequence -- keep old representative strains
metadata_df = metadata_df.merge(new_metadata_df, on='representative_strain_ha_sequence', how='left')

# Make repool dataframe using chosen well 
repool_df = (mean_single_well
             .query('mean_ratio_to_add != inf')
             .query(f'well == "{well2}"')
             [['strain','fraction_strain','mean_ratio_to_add']]
             .assign(x_volume_to_add = lambda x: x['mean_ratio_to_add'] * repool_factor)
             .merge(lib_id_df, how='outer')
             .assign(
                 subtype = lambda x: x['name'].str.replace('flu-seqneut-25to26_','').str.split('_').str[0],
                 number = lambda x: pd.to_numeric(
                     x['name'].str.replace('flu-seqneut-25to26_','').str.split('_').str[1],  # removed inner lambda
                     errors='coerce').fillna(1e6).astype(int),  # use a big number to push NaNs to bottom
                 strain_id = lambda x: x['subtype'] + '_' + x['number'].astype(str)
             )
             .sort_values(by=['subtype', 'number'], ascending=True)
             .drop(columns=['number', 'name', 'barcode', 'nt_sequence_HA_ectodomain','protein_sequence_HA_ectodomain'])  # number is now temporary
             .drop_duplicates()
             .dropna(subset=['fraction_strain'])
             .reset_index(drop=True)
)

pooling_mathdir = '../results/pooling_math'
os.makedirs(pooling_mathdir, exist_ok=True)

# drop strains
strains_to_drop = [
    'A/Bangkok/P2277/2025_H1N1', # potential extremely low level plasmid contamination
    'A/Chinautla/FLU-331/2025_H1N1', # low titer
    'A/SouthAustralia/2518902563/2025_H1N1', # low titer
    'A/Victoria/1493/2025_H1N1', # low titer
    'A/Queensland/159/2025_H1N1', # low titer
    'A/BritishColumbia/PHL-2478/2025_H1N1', # low titer
    'A/SouthAustralia/2518911830/2025_H1N1', # low titer    
    'A/Lao/820-2765/2025_H1N1', # low titer
    'A/Alagoas/4131/2025_H1N1', # low titer
    'A/NewSouthWales/2525913630/2025_H1N1', # low titer
    'A/Catalonia/NSVH102686666/2025_H1N1', # low titer 
    'A/SouthAustralia/2521506017/2025_H1N1', # low titer
    'A/SouthAustralia/2521715152/2025_H1N1', # low titer 
]
trimmed_repool_df = repool_df[~repool_df['strain'].isin(strains_to_drop)]

# Merge with metadata
trimmed_repool_df = trimmed_repool_df.merge(metadata_df[['old_derived_haplotype', 'derived_haplotype','strain','count','latest_sequence', 'representative_strain_ha_sequence']], on='strain', how='left')

# Save metadata
# trimmed_repool_df.to_csv('../results/flu-seqneut-2025_viral_library_old_metadata.csv',index=False)

# Save pooling math
pooling_df = trimmed_repool_df.sort_values(by='x_volume_to_add', ascending=False).reset_index(drop=True)[['strain_id','x_volume_to_add','strain']]
pooling_df['number'] = pooling_df['strain_id'].str.split('_').str[1].fillna(1e6).astype(int)
pooling_df['subtype'] = pooling_df['strain_id'].str.split('_').str[0]
pooling_df = (pooling_df
    .sort_values(by=['subtype','number'], ascending=True)
    .drop(columns=['number', 'subtype'])
             )
# pooling_df.to_csv('../results/pooling_math/2026-01-12_repooling_math.csv',index=False)

# Save info on dropped strains
dropped_repool_df = repool_df[repool_df['strain'].isin(strains_to_drop)]
# dropped_repool_df.to_csv('../results/pooling_math/dropped_strains.csv',index=False)

# Summarize top 10 lowest titer strains
print('Displaying the top 10 lowest titer strains that made the cut...')
trimmed_repool_df.sort_values(by='x_volume_to_add', ascending=False).head(10).reset_index(drop=True)


Displaying the top 10 lowest titer strains that made the cut...


,strain,fraction_strain,mean_ratio_to_add,x_volume_to_add,accession,strain_type,subtype,subclade,num_date,vaccine_type,strain_id,old_derived_haplotype,derived_haplotype,count,latest_sequence,representative_strain_ha_sequence
0,A/Thailand/8/2022_H3N2,0.000147,27.942964,698.574092,NaN,vaccine,NaN,J,2022.997,egg,NaN,NaN,NaN,NaN,NaN,NaN
1,A/Massachusetts/18/2022_H3N2,0.000672,15.248708,381.217689,NaN,vaccine,NaN,J,2022.422,cell,NaN,NaN,NaN,NaN,NaN,NaN
2,A/Croatia/10136RV/2023-egg_H3N2,0.001165,11.610838,290.270952,PV262858_D202A,vaccine,NaN,J.2,2023.923,egg,NaN,NaN,NaN,NaN,NaN,NaN
3,A/Darwin/1461/2025_H3N2,0.001087,9.652986,241.324640,PV498778_R78K,circulating_2025to2026,H3N2,NaN,NaN,NaN,H3N2_47,"J.2.2:G62K,S145N,P239Q",NaN,NaN,NaN,MKAIIALSNILCLVFAQKIPGNDNSTATLCLGHHAVPNGTIVKTIT...
4,A/SouthAfrica/PET37604/2025_H3N2,0.001413,9.537839,238.445968,PQ422645_P71T_N175S_S215T_G402E,circulating_2025to2026,H3N2,NaN,NaN,NaN,H3N2_52,"J.2.2:P55T,T65K,N159S,S199T,I214T","J.2.2:P55T,T65K,N159S,S199T,I214T",2.0,2025-07-08,MKAIIALSNILCLVFAQKIPGNDNSTATLCLGHHAVPNGTIVKTIT...
5,A/Tasmania/2521815682/2025_H3N2,0.001150,9.341037,233.525925,PX422988_K18R,circulating_2025to2026,H3N2,NaN,NaN,NaN,H3N2_12,"J.2.4.1:N2R,F79V,R173Q","J.2.4:K2R,F79V,S144N,N158D,I160K,T328A",1.0,2025-08-06,MKAIIALSNILCLVFAQRIPGNDNSTATLCLGHHAVPNGTIVKTIT...
6,A/England/174/2025_H3N2,0.001853,8.120599,203.014967,PV538018,circulating_2025to2026,H3N2,NaN,NaN,NaN,H3N2_23,J.2.2:T65K,NaN,NaN,NaN,MKAIIALSNILCLVFAQKIPGNDNSTATLCLGHHAVPNGTIVKTIT...
7,A/Indonesia/BIOKES-INAD491/2025_H3N2,0.001451,7.838080,195.951998,PV584644_S160G_S416A,circulating_2025to2026,H3N2,NaN,NaN,NaN,H3N2_36,"J.2.2:T65K,S144G",NaN,NaN,NaN,MKAIIALSNILCLVFAQKIPGNDNSTATLCLGHHAVPNGTIVKTIT...
8,A/DistrictOfColumbia/27/2023_H3N2,0.001320,6.118594,152.964848,NaN,vaccine,NaN,J.2,2023.936,cell,NaN,NaN,NaN,NaN,NaN,NaN
9,A/Bangkok/P2390/2025_H3N2,0.000964,4.726421,118.160522,PX422972,circulating_2025to2026,H3N2,NaN,NaN,NaN,H3N2_3,J.2.4.1:A272T,NaN,NaN,NaN,MKAIIALSNILCLVFAQNIPGNDNSTATLCLGHHAVPNGTIVKTIT...


In [17]:
print(f'Adding {repool_factor}x of each strain ratio...')
_sum = (
    sum(trimmed_repool_df.mean_ratio_to_add) * repool_factor
     )
print(_sum, 'uL total pool')
print('This means adding strains in volumes ranging from...')
print(trimmed_repool_df.x_volume_to_add.min(), 'uL to ', trimmed_repool_df.x_volume_to_add.max(), 'uL')
print('Assuming worse case scenario of 1:16 on 150k cells...')
volume_per_plate = (50/16)*110
print(volume_per_plate, 'uL per plate')
print('About how many plates can I run?')
print((_sum-10)/volume_per_plate)

Adding 25x of each strain ratio...
6918.353912108404 uL total pool
This means adding strains in volumes ranging from...
12.199227741003197 uL to  698.5740919922683 uL
Assuming worse case scenario of 1:16 on 150k cells...
343.75 uL per plate
About how many plates can I run?
20.097029562497177


## Visualize barcode- and strain-level balancing in the current pool

In [18]:
# Plot the current fraction of each strain in the pool
strains_chart = (
    alt.Chart(mean_single_well.query('fraction_strain!=0'))
    .encode(
        alt.X(
            "fraction_strain",
            scale=alt.Scale(nice=False, padding=3),
        ),
        alt.Y("strain"),
        
        tooltip = ['strain', 'fraction_strain', 'est_tcid50'],
    )
).mark_bar(height={"band": 0.85}).properties(
        height=alt.Step(10),
        width=250,
        title="",
    ).properties(
        height = alt.Step(10),
        width = 200,
        title = "Strain representation, repool")

# add veritcal line where we would expect equal representation of all barcodes in pool
expected_line = alt.Chart(
    pd.DataFrame({'x': [1/num_strains]})
).mark_rule(strokeDash = [2,2], strokeWidth = 2).encode(x = 'x')

# plot both barcode counts and expected line
strains_chart + expected_line

alt.LayerChart(...)

In [19]:
# Each barcode fraction across strains
all_barcode_counts = counts[['strain', 'barcode', 'count', 'well']].dropna()
single_well_all_barcode_counts = all_barcode_counts[all_barcode_counts['well'].isin([f'{well1}',f'{well2}'])]

# Get tidy single well means
tidy_single_well = single_well_all_barcode_counts[['strain','barcode','count']].groupby(['strain', 'barcode']).mean().reset_index()
# Get sums for each strain
strain_sums_df = tidy_single_well.groupby('strain').sum().rename(columns = {'count': 'strain_count_sum'}).reset_index()
# Merge and calculate per strain the fraction represented by each barcode
tidy_single_well = tidy_single_well.merge(strain_sums_df[['strain', 'strain_count_sum']], 
                       on = ['strain'],
                       validate="many_to_one",
                      )
tidy_single_well['per_strain_fraction_barcode'] = tidy_single_well['count'] / tidy_single_well['strain_count_sum']
tidy_single_well['barcode_letter'] = tidy_single_well.groupby('strain').cumcount().apply(lambda x: string.ascii_uppercase[x])

# Plot as colored bar chart
bar_chart = alt.Chart(tidy_single_well).mark_bar(height={"band": 0.85}).encode(
    x = 'per_strain_fraction_barcode',
    y = 'strain',
    color=alt.Color('barcode_letter', legend=None).scale(range=color_palette),
    tooltip = ['strain', 'per_strain_fraction_barcode', 'barcode'],
).configure_axis(grid=False).properties(
        height = alt.Step(10),
        width = 200,
        title = "Barcode fraction for each strain, repool")

bar_chart

alt.Chart(...)